# SIGMA-SED: variational autoencoder and mixture segmentation

Compact method notebook for TEM users: normalized 4D-STEM diffraction
patterns → convolutional variational autoencoder (VAE) → latent means →
Bayesian Gaussian-mixture segmentation → cluster maps and mean diffraction
patterns. The encoder processes **2D patterns**; its latent representation
need not be two-dimensional.

Supply an already centred, preprocessed and normalized SED signal of shape
`(scan_y, scan_x, detector_y, detector_x)`. Raw data, trained weights, specimen
identifiers and acquisition calibration values are not distributed here.
Clusters describe diffraction similarity; phase/orientation assignment
requires subsequent crystallographic validation.

**Licence.** This notebook derives from GPLv3 code and is released under
GPLv3, not the repository's MIT licence. See
[`LICENSE_04_sigma_sed.txt`](LICENSE_04_sigma_sed.txt).


## Environment

This notebook uses the [SIGMA-SED fork](https://github.com/CheukHinHoJerry/sigma-sed),
specifically `SEDDataset`, `VariationalAutoEncoder2D`, `Experiment` and
`PixelSegmenter`. Installing the base `emsigma` distribution alone may not
provide these SED extensions.

The inspected source checkout specifies Python ≥3.10, PyTorch 2.0.1,
HyperSpy 1.7.5, pyxem 0.16, NumPy 1.24.4 and scikit-learn 1.3.0.
Use a separate Python 3.10 environment for this legacy fork; compatibility
with the latest HyperSpy/pyxem releases has not been established.

Setup, following the source checkout's README:

```bash
conda create -n sigma-sed python=3.10
conda activate sigma-sed
git clone https://github.com/CheukHinHoJerry/sigma-sed.git
cd sigma-sed
python -m pip install .
python -m pip check
jupyter lab
```

Use a fork revision containing all four interfaces above. These instructions
are based on the supplied checkout, not a newly tested dependency lock.
`Experiment` selects CUDA when available and otherwise uses CPU; CPU training
will be slower. Record the fork revision and installed versions for each run.

## 1. Configuration and input

Run from the folder containing this notebook, with your signal under `data/`.
All paths below are generic. **`None` is a redacted/unset value**, not a
physical zero. Supply the accelerating voltage and check the input signal's
detector scale, origin and units before azimuthal integration. Optional
plotting calibration must match the preprocessed detector grid; otherwise
diffraction plots use detector-pixel coordinates.

Training and mixture settings are editable starting points, not calibrated
instrument settings or validated choices for another dataset. Input values
must already lie in [0, 1] for the VAE's sigmoid decoder. No additional
normalization is applied here.

In [ ]:
from pathlib import Path

# Repository-relative paths. This works when Jupyter starts from either
# the repository root or its notebooks/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = REPO_ROOT / "data/input_sed.zspy"
OUTPUT_ROOT = REPO_ROOT / "outputs/sigma_sed"
MODE = "train"  # "train" or "load"
CHECKPOINT_PATH = None  # Local checkpoint FILE, not its containing directory.

# Acquisition calibration: intentionally unset for public distribution.
BEAM_ENERGY_KEV = None
RECIPROCAL_PIXEL_SIZE_A_INV = None  # Optional isotropic plotting scale.
DIFFRACTION_ORIGIN_PX = None  # Optional (x, y) in the preprocessed detector grid.
INTEGRATION_UNIT = "k_A^-1"
RADIAL_BINS = 70

# VAE architecture and optimization.
RANDOM_SEED = 0
HIDDEN_CHANNELS = (128, 64, 32)
LATENT_DIM = 25
TRAINING = dict(
    num_epochs=40, batch_size=32, learning_rate=2e-4,
    weight_decay=0.0, task="train_all", criterion="MSE", KLD_lambda=1e-4,
)
LATENT_BATCH_SIZE = 128

# Bayesian mixture: component count is an upper bound, not a phase count.
MIXTURE = dict(
    n_components=20, random_state=RANDOM_SEED, init_params="kmeans",
    covariance_type="full", n_init=1, max_iter=100, reg_covar=1e-6,
)
PLOT_MAX_POINTS = 10000
SELECTED_CLUSTER = None  # None selects the most populated cluster.
DP_VMAX = None  # None uses one shared maximum across occupied-cluster means.
RADIAL_BIN_RANGE = None  # Optional (start, stop), in bin indices; stop excluded.
SHOW_FIGURES = True  # Overview, map and one selected cluster only.
SAVE_FIGURES = True  # PNG overviews and diagnostics for every occupied cluster.
FIGURE_DPI = 180
SAVE_RESULTS = False  # Training still writes checkpoints to a new run folder.

In [ ]:
import json
import tempfile
import warnings
from importlib.metadata import PackageNotFoundError, version

import matplotlib.pyplot as plt
import numpy as np
import pyxem  # Registers diffraction signal classes with HyperSpy.
import torch
from matplotlib.colors import BoundaryNorm
from sklearn.decomposition import PCA

from sigma.models.autoencoder import VariationalAutoEncoder2D
from sigma.src.dim_reduction import Experiment
from sigma.src.segmentation import PixelSegmenter
from sigma.src.utils import same_seeds
from sigma.utils.loadsed import SEDDataset

# The legacy fork disables warnings at import time; restore normal reporting.
warnings.simplefilter("default")
same_seeds(RANDOM_SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
package_versions = {}
for package in ("emsigma", "torch", "hyperspy", "pyxem", "numpy", "scikit-learn"):
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = "source checkout / version unavailable"
print(package_versions)

In [ ]:
if MODE not in {"train", "load"}:
    raise ValueError("MODE must be 'train' or 'load'.")
if BEAM_ENERGY_KEV is None:
    raise ValueError("Set BEAM_ENERGY_KEV from your acquisition metadata.")
if not np.isfinite(BEAM_ENERGY_KEV) or BEAM_ENERGY_KEV <= 0:
    raise ValueError("BEAM_ENERGY_KEV must be positive and finite.")
if not DATA_PATH.exists():
    raise FileNotFoundError("Set DATA_PATH to your preprocessed SED signal.")
if MODE == "load" and (CHECKPOINT_PATH is None or not Path(CHECKPOINT_PATH).is_file()):
    raise ValueError("Set CHECKPOINT_PATH to a compatible model checkpoint file.")
if (RECIPROCAL_PIXEL_SIZE_A_INV is None) != (DIFFRACTION_ORIGIN_PX is None):
    raise ValueError("Set both plotting scale and diffraction origin, or leave both None.")
if RECIPROCAL_PIXEL_SIZE_A_INV is not None:
    if not np.isfinite(RECIPROCAL_PIXEL_SIZE_A_INV) or RECIPROCAL_PIXEL_SIZE_A_INV <= 0:
        raise ValueError("The reciprocal-space plotting scale must be positive and finite.")
    if len(DIFFRACTION_ORIGIN_PX) != 2 or not np.isfinite(DIFFRACTION_ORIGIN_PX).all():
        raise ValueError("The diffraction origin must contain two finite coordinates.")
if LATENT_DIM < 2 or len(HIDDEN_CHANNELS) != 3 or min(HIDDEN_CHANNELS) < 1:
    raise ValueError("Use at least two latent features and three positive channel sizes.")
if min(RADIAL_BINS, LATENT_BATCH_SIZE, PLOT_MAX_POINTS, MIXTURE["n_components"]) < 1:
    raise ValueError("Bin, batch, plot and component counts must be positive.")
if DP_VMAX is not None and (not np.isfinite(DP_VMAX) or DP_VMAX <= 0):
    raise ValueError("DP_VMAX must be positive and finite, or None for a shared automatic limit.")
if FIGURE_DPI <= 0:
    raise ValueError("FIGURE_DPI must be positive.")

In [ ]:
sed_ds = SEDDataset(
    str(DATA_PATH), n_root=1, integral_type="azimuthal",
    unit=INTEGRATION_UNIT, beam_energy=BEAM_ENERGY_KEV, npt=RADIAL_BINS,
)
if getattr(sed_ds.base_dataset, "_lazy", False):
    sed_ds.base_dataset.compute()
patterns = np.asarray(sed_ds.base_dataset.data, dtype=np.float32)
if patterns.ndim != 4:
    raise ValueError("Expected (scan_y, scan_x, detector_y, detector_x).")
scan_shape = patterns.shape[:2]
detector_shape = patterns.shape[-2:]
image_size = detector_shape[0]
if detector_shape[0] != detector_shape[1] or image_size < 8 or image_size % 8:
    raise ValueError("This VAE requires square detector images with size divisible by 8.")
if not np.isfinite(patterns).all() or patterns.min() < 0 or patterns.max() > 1:
    raise ValueError("Provide finite diffraction intensities already normalized to [0, 1].")
if patterns.max() == patterns.min():
    raise ValueError("The input is constant; no diffraction contrast is available.")
n_patterns = int(np.prod(scan_shape))
if n_patterns < 2 or MIXTURE["n_components"] > n_patterns:
    raise ValueError("Provide at least two scan points and no more components than points.")
sed_ds.base_dataset.data = patterns
print(f"Scan shape: {scan_shape}; detector shape: {detector_shape}")

### Output folder

Each run uses a fresh directory under `OUTPUT_ROOT`. `SAVE_FIGURES=True`
saves readable PNG diagnostics; `SHOW_FIGURES` controls inline display.
Figures regenerated within a run receive a suffix instead of overwriting
an earlier image. Model checkpoints are also stored inside this run.

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run_", dir=OUTPUT_ROOT))
figure_dir = run_dir / "figures"
if SAVE_FIGURES:
    figure_dir.mkdir()

def present_figure(fig, stem, *, show=True, save=True):
    if SAVE_FIGURES and save:
        figure_dir.mkdir(exist_ok=True)
        suffix = 0
        while True:
            filename = f"{stem}.png" if suffix == 0 else f"{stem}_{suffix:02d}.png"
            target = figure_dir / filename
            try:
                stream = target.open("xb")
                break
            except FileExistsError:
                suffix += 1
        with stream:
            fig.savefig(stream, format="png", dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
    if SHOW_FIGURES and show:
        plt.show()
    plt.close(fig)

print(f"Run folder: {run_dir}")

### Input check

The summed-intensity navigator retains array row/column order. The radial
profile is an integration diagnostic; the VAE is trained on full 2D patterns.
Its horizontal axis is shown as bin index to avoid implying an undisclosed
physical calibration.

In [ ]:
radial_profiles = np.asarray(sed_ds.spectra.data)
if radial_profiles.shape[:2] != scan_shape or not np.isfinite(radial_profiles).all():
    raise ValueError("Check azimuthal integration and input calibration.")
navigator = patterns.sum(axis=(-2, -1))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
im = axes[0].imshow(navigator, cmap="gray", origin="upper")
axes[0].set(title="Integrated diffraction intensity", xlabel="Scan x (pixel)", ylabel="Scan y (pixel)")
fig.colorbar(im, ax=axes[0], label="Summed normalized intensity (a.u.)")
axes[1].plot(radial_profiles.mean(axis=(0, 1)), color="#285780")
axes[1].set(title="Mean radial profile", xlabel="Radial bin", ylabel="Intensity (a.u.)")
present_figure(fig, "input_overview")

## 2. Learn diffraction representations

The VAE minimizes reconstruction MSE plus a weighted KL-divergence term.
`train_all` uses all scan points; the inspected fork uses an 80/20 random
split for `train_eval` (not a spatially independent validation split).
Seeds improve repeatability within a fixed software/hardware environment.

Each run gets a fresh output directory because `Experiment` saves model
checkpoints automatically. In `load` mode, supply a trusted checkpoint file
with the same architecture and input normalization; training is skipped.

In [ ]:
same_seeds(RANDOM_SEED)
ex = Experiment(
    descriptor="sed_vae", general_results_dir=str(run_dir),
    model=VariationalAutoEncoder2D,
    model_args=dict(hidden_layer_sizes=HIDDEN_CHANNELS, lat_dim=LATENT_DIM, img_size=image_size),
    chosen_dataset=patterns, save_model_every_epoch=False,
)
if MODE == "train":
    ex.run_model(**TRAINING)
else:
    # map_location also permits a CUDA-trained checkpoint to be used on CPU.
    checkpoint = torch.load(str(CHECKPOINT_PATH), map_location=ex.device)
    ex.model.load_state_dict(checkpoint["params"])
print(f"Device: {ex.device}; run folder: {run_dir.name}")

### Encode the scan

Extract the encoder's latent **means** in scan order, with an explicit batch
size to bound device memory. This uses the same `_encode` method as the
inspected fork's `get_latent`, without its fixed large inference batch.

In [ ]:
flat_patterns = patterns.reshape(n_patterns, 1, *detector_shape)
latent_parts = []
ex.model.eval()
with torch.no_grad():
    for start in range(0, n_patterns, LATENT_BATCH_SIZE):
        batch = torch.from_numpy(flat_patterns[start:start + LATENT_BATCH_SIZE]).to(ex.device)
        latent_parts.append(ex.model._encode(batch).detach().cpu().numpy())
latent = np.concatenate(latent_parts, axis=0)
if latent.shape != (n_patterns, LATENT_DIM) or not np.isfinite(latent).all():
    raise ValueError("Invalid latent representation; inspect training and input data.")
print(f"Latent array: {latent.shape}")

## 3. Segment and inspect the scan

Fit a Bayesian Gaussian mixture in the **full latent space**. The component
bound may exceed the number of occupied clusters; empty components are
excluded from pattern summaries and export. Inspect convergence and cluster
occupancy before interpreting the result. Membership probabilities describe
the fitted mixture, not independently calibrated phase confidence.

The exploratory BIC sweep and alternative clustering trials have been removed.
For the distinction between classical GMM/BIC and the Bayesian component
bound, see the [scikit-learn mixture guide](https://scikit-learn.org/stable/modules/mixture.html).

In [ ]:
# The legacy helper averages every candidate component, including empty ones.
# Suppress only its empty-mean warnings; convergence warnings remain visible.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Mean of empty slice.*", category=RuntimeWarning)
    warnings.filterwarnings("ignore", message="invalid value encountered in divide", category=RuntimeWarning)
    ps = PixelSegmenter(
        latent=latent, dataset=sed_ds, method="BayesianGaussianMixture",
        method_args=MIXTURE,
    )
if not ps.model.converged_:
    warnings.warn("Mixture did not converge: review max_iter, regularization and component count.")
labels = np.asarray(ps.labels, dtype=np.int32)
occupied_ids, cluster_counts = np.unique(labels, return_counts=True)
cluster_map = labels.reshape(scan_shape)
mean_patterns = ps.mu[occupied_ids].reshape(-1, *detector_shape)
mean_radial_profiles = ps.mu1d[occupied_ids]
if not np.isfinite(mean_patterns).all():
    raise ValueError("Non-finite mean pattern in an occupied cluster.")
if mean_radial_profiles.ndim != 2 or not np.isfinite(mean_radial_profiles).all():
    raise ValueError("Invalid radial profile in an occupied cluster.")
dp_vmax = float(mean_patterns.max()) if DP_VMAX is None else DP_VMAX
radial_slice = slice(None)
if RADIAL_BIN_RANGE is not None:
    start_bin, stop_bin = RADIAL_BIN_RANGE
    if (not isinstance(start_bin, int) or not isinstance(stop_bin, int)
            or not 0 <= start_bin < stop_bin <= mean_radial_profiles.shape[1]):
        raise ValueError("RADIAL_BIN_RANGE must select a non-empty interval of available bins.")
    radial_slice = slice(start_bin, stop_bin)
radial_limits = (float(mean_radial_profiles[:, radial_slice].min()),
                 float(mean_radial_profiles[:, radial_slice].max()))
radial_padding = max((radial_limits[1] - radial_limits[0]) * 0.05, 1e-8)
for cluster_id, count in zip(occupied_ids, cluster_counts):
    print(f"Cluster {cluster_id}: {count} scan points ({count / n_patterns:.1%})")

### Latent projection and cluster map

PCA provides a 2D display when the latent dimension exceeds two; it is not
used for mixture fitting. The scatter may be subsampled for readability.
The map includes every scan point, with the same cluster colours as the
scatter. Check spatial coherence against the diffraction-pattern evidence.

In [ ]:
if latent.shape[1] > 2:
    display_pca = PCA(n_components=2, svd_solver="full")
    latent_display = display_pca.fit_transform(latent)
    projection_title = f"PCA display ({display_pca.explained_variance_ratio_.sum():.1%} variance)"
    latent_axis_labels = ("Latent PC1", "Latent PC2")
else:
    latent_display = latent
    projection_title = "Latent representation"
    latent_axis_labels = ("Latent feature 1", "Latent feature 2")
rng = np.random.default_rng(RANDOM_SEED)
shown = np.sort(rng.choice(n_patterns, min(PLOT_MAX_POINTS, n_patterns), replace=False))
display_labels = np.searchsorted(occupied_ids, labels)
cluster_cmap = plt.get_cmap("tab20", len(occupied_ids))
cluster_norm = BoundaryNorm(np.arange(len(occupied_ids) + 1) - 0.5, len(occupied_ids))

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].scatter(
    latent_display[shown, 0], latent_display[shown, 1],
    c=display_labels[shown], cmap=cluster_cmap, norm=cluster_norm,
    s=3, alpha=0.6, linewidths=0, rasterized=True,
)
axes[0].set(title=projection_title, xlabel=latent_axis_labels[0], ylabel=latent_axis_labels[1])
im = axes[1].imshow(
    display_labels.reshape(scan_shape), cmap=cluster_cmap, norm=cluster_norm,
    origin="upper", interpolation="nearest",
)
axes[1].set(title="Diffraction-similarity clusters", xlabel="Scan x (pixel)", ylabel="Scan y (pixel)")
bar = fig.colorbar(im, ax=axes[1], ticks=np.arange(len(occupied_ids)), label="Cluster ID")
bar.ax.set_yticklabels(occupied_ids)
present_figure(fig, "latent_cluster_map")

### Cluster diagnostics

This three-panel view retains the basic SED diagnostics of the original
`gui.show_cluster_distribution`: membership probability, mean diffraction
pattern and mean radial profile. It uses ordinary Matplotlib figures so the
same view can be inspected inline or saved without a widget session.

`SELECTED_CLUSTER=None` previews the most populated cluster. `DP_VMAX`
controls the diffraction display ceiling (the original `maxint` role),
while `RADIAL_BIN_RANGE` selects an index interval of the radial profile.
All cluster diagnostics share intensity limits to support comparison.

The diffraction pattern is the arithmetic
**mean of preprocessed input patterns**, not a raw-count sum or a decoded VAE
pattern. Pixel axes are used unless both plotting calibration fields are set.
The calibrated view assumes an isotropic scale and preserves detector row order.

In [ ]:
def plot_cluster_diagnostics(cluster_id):
    if cluster_id not in occupied_ids:
        raise ValueError("Choose an occupied cluster ID.")
    pattern_index = int(np.flatnonzero(occupied_ids == cluster_id)[0])
    dp = mean_patterns[pattern_index]
    extent = None
    axis_labels = ("Detector x (pixel)", "Detector y (pixel)")
    if RECIPROCAL_PIXEL_SIZE_A_INV is not None:
        origin_x, origin_y = DIFFRACTION_ORIGIN_PX
        height, width = detector_shape
        scale = RECIPROCAL_PIXEL_SIZE_A_INV
        extent = [(-0.5 - origin_x) * scale, (width - 0.5 - origin_x) * scale,
                  (height - 0.5 - origin_y) * scale, (-0.5 - origin_y) * scale]
        axis_labels = ("kx (Å⁻¹)", "ky (Å⁻¹; detector-row direction)")

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), layout="constrained")
    fig.suptitle(f"Cluster {cluster_id} | {cluster_counts[pattern_index]} scan points")
    membership = ps.prob_map[:, cluster_id].reshape(scan_shape)
    im = axes[0].imshow(membership, cmap="viridis", vmin=0, vmax=1, origin="upper")
    axes[0].set(title="Membership probability", xlabel="Scan x (pixel)", ylabel="Scan y (pixel)")
    fig.colorbar(im, ax=axes[0], label="Mixture membership probability")
    im = axes[1].imshow(dp, cmap="viridis", vmin=0, vmax=dp_vmax, origin="upper", extent=extent)
    axes[1].set(title="Mean diffraction pattern", xlabel=axis_labels[0], ylabel=axis_labels[1])
    fig.colorbar(im, ax=axes[1], label="Mean normalized intensity (a.u.)")
    radial_bins = np.arange(mean_radial_profiles.shape[1])[radial_slice]
    axes[2].plot(radial_bins, mean_radial_profiles[pattern_index, radial_slice], color="#285780")
    axes[2].set(title="Mean radial profile", xlabel="Radial bin", ylabel="Intensity (a.u.)",
                ylim=(radial_limits[0] - radial_padding, radial_limits[1] + radial_padding))
    return fig

In [ ]:
cluster_id = int(occupied_ids[np.argmax(cluster_counts)]) if SELECTED_CLUSTER is None else SELECTED_CLUSTER
fig = plot_cluster_diagnostics(cluster_id)
# The following cell saves all occupied clusters, including this one.
present_figure(fig, f"cluster_{int(cluster_id):03d}_diagnostics", save=False)

### Save every occupied cluster for review

With `SAVE_FIGURES=True`, inspect `run_…/figures/` under `OUTPUT_ROOT`:
`input_overview.png`, `latent_cluster_map.png` and one
`cluster_###_diagnostics.png` per occupied cluster. Each image has the same
three diagnostic panels as the selected-cluster preview. Cluster IDs are
shown in titles and the external map colour bar only.

Check the saved cluster views for coherent scan locations, meaningful
diffraction contrast and consistent radial features before interpretation.

In [ ]:
if SAVE_FIGURES:
    for cluster_id in occupied_ids:
        fig = plot_cluster_diagnostics(int(cluster_id))
        present_figure(fig, f"cluster_{int(cluster_id):03d}_diagnostics", show=False)
    print(f"Saved {len(occupied_ids)} occupied-cluster diagnostics. Review: {figure_dir}")
else:
    print("Figure saving disabled; set SAVE_FIGURES=True to create the PNG review set.")

### Optional export

Set `SAVE_RESULTS=True` to save the label map, latent means, cluster counts
and occupied-cluster mean patterns in a new subdirectory of this run. TXT
patterns contain intensity matrices only. The JSON file records array order
and plotting calibration (null when unset), without input/checkpoint paths.
No directory is cleared and existing exports are never overwritten.

In [ ]:
if SAVE_RESULTS:
    export_dir = Path(tempfile.mkdtemp(prefix="export_", dir=run_dir))
    np.save(export_dir / "cluster_labels.npy", cluster_map)
    np.save(export_dir / "latent_means.npy", latent)
    np.savetxt(
        export_dir / "cluster_counts.csv", np.column_stack((occupied_ids, cluster_counts)),
        delimiter=",", fmt="%d", header="cluster_id,scan_points", comments="",
    )
    for cluster_id, mean_dp in zip(occupied_ids, mean_patterns):
        np.savetxt(export_dir / f"cluster_{int(cluster_id):03d}_mean_dp.txt", mean_dp)
    run_metadata = dict(
        model="VariationalAutoEncoder2D", mode=MODE, seed=RANDOM_SEED,
        hidden_channels=list(HIDDEN_CHANNELS), latent_dim=LATENT_DIM,
        training=TRAINING if MODE == "train" else None, mixture=MIXTURE,
        scan_shape=list(scan_shape), detector_shape=list(detector_shape),
        array_order=["scan_y", "scan_x", "detector_y", "detector_x"],
        beam_energy_kev=BEAM_ENERGY_KEV,
        reciprocal_pixel_size_a_inv=RECIPROCAL_PIXEL_SIZE_A_INV,
        diffraction_origin_px=DIFFRACTION_ORIGIN_PX,
        mean_pattern_definition="Arithmetic mean of normalized preprocessed input patterns",
        package_versions=package_versions,
    )
    with (export_dir / "run_metadata.json").open("x") as stream:
        json.dump(run_metadata, stream, indent=2, allow_nan=False)
    print(f"Exported {len(occupied_ids)} occupied clusters to {export_dir.name}")
else:
    print("Result export disabled; set SAVE_RESULTS=True to save analysis products.")

## Review and attribution

Before reuse, check preprocessing/normalization, reconstruction quality,
mixture convergence, cluster occupancy and representative diffraction
patterns. Compare parameter/seed choices before assigning physical meaning.
The supplied notebook is a method template, not an executed demonstration;
run all cells after supplying your own data and calibration.

This notebook derives from the
[SIGMA-SED fork](https://github.com/CheukHinHoJerry/sigma-sed) of
[SIGMA](https://github.com/poyentung/sigma), which is distributed under the
GNU General Public License v3.0. It is therefore released under GPLv3 rather
than the repository's MIT licence; the full terms are in
[`LICENSE_04_sigma_sed.txt`](LICENSE_04_sigma_sed.txt).
For SIGMA method attribution, see [Tung et al.](https://doi.org/10.1029/2022GC010530).

This is a reduced public version of a working notebook. Experimental
identifiers, acquisition calibration values and saved outputs are not
distributed; the VAE and Bayesian-mixture method is unchanged.